# Démonstration d'appel à l'anonymiseur

Ce notebook montre deux manières d'utiliser l'anonymisation :

1. Appel direct de la fonction Python `anonymize_text` (pas besoin de serveur Flask)
2. (Optionnel) Appel HTTP de l'API Flask (`main_eval.py`)

Modes de politique disponibles (simplifiés) :
- `L0` : sans LLM (regex + NER éventuel)
- `L1` (ou toute autre valeur) : avec LLM (détection/paraphrase/audit) si une clé OpenRouter est configurée.

> Pour activer le mode LLM, exporte ta clé : `export OPENROUTER_API_KEY="sk-..."` avant d'exécuter le code.


In [1]:
# Appel direct (mode sans LLM : L0)
import sys, pathlib
sys.path.append(str(pathlib.Path().resolve()))

from src.orchestrator import anonymize_text

sample_text = "Mon email est jean.dupont@example.com et mon IP est 192.168.1.10"

result_L0 = anonymize_text(
    value=sample_text,
    scope_id="demo-scope-1",
    secret_salt="demo-secret",
    level="L0",  # désactive LLM
    openrouter_models=None,
    ner_results=[]  # tu peux injecter ici des entités NER déjà détectées
)

result_L0

{'text': 'Mon email est [HOST_JHC]@[HOST_PGY] et mon IP est [IP_BTF]',
 'audit': {'entities': [{'start': 52,
    'end': 64,
    'replacement': '[IP_BTF]',
    'etype': 'IP',
    'surface': '192.168.1.10',
    'source': 'regex'},
   {'start': 26,
    'end': 37,
    'replacement': '[HOST_PGY]',
    'etype': 'HOST',
    'surface': 'example.com',
    'source': 'regex'},
   {'start': 14,
    'end': 25,
    'replacement': '[HOST_JHC]',
    'etype': 'HOST',
    'surface': 'jean.dupont',
    'source': 'regex'}],
  'risk': {'risk_score': 0, 'findings': [], 'recommendations': []},
  'llm_error': None,
  'llm_used': False,
  'models': None},
 'policy': {'level': 'L0',
  'placeholder_style': 'typed',
  'scope': 'ticket',
  'mapping_retention': 'session',
  'date_granularity': 'month',
  'time_granularity': 'none',
  'location_granularity': 'city',
  'ip_policy': 'public_private',
  'amount_binning': 'none',
  'org_policy': 'categorize',
  'llm_detection': False,
  'llm_paraphrase': False,
  'llm_a

In [2]:
!export OPENROUTER_API_KEY=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

In [1]:
# Appel direct (mode avec LLM : L1) — nécessite OPENROUTER_API_KEY défini
import os
from src.orchestrator import anonymize_text
from dotenv import load_dotenv

/opt/anaconda3/envs/dpv/lib/python3.11/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:


# Appel direct (mode avec LLM : L1) — nécessite OPENROUTER_API_KEY défini

load_dotenv()

if os.getenv("OPENROUTER_API_KEY"):
    sample_text_llm = "Support chat: User Jane Doe, marketing manager in Seattle branch of GlobalInc, reported issue with internal tool version 4.2.1 on 2024-05-15. Her assistant, based in the same office, added that it's similar to what happened to the finance team in Chicago last quarter. Email: jane.doe@globalinc.com. Rare combo: marketing role + branch + tool version + cross-team reference. Amount affected: $25000. He noted the MAC address 00:11:22:33:44:55 and IP 10.0.0.1."
    result_L1 = anonymize_text(sample_text_llm, "scope", "secret", level="L1", overrides={"llm_paraphrase": False})
    print(result_L1["text"])
    print(result_L1["audit"]["llm_error"])
    [e for e in result_L1["audit"]["entities"] if e.get("source")=="llm-edit"]  # doit être []
else:
    print("Aucune clé OPENROUTER_API_KEY trouvée: le mode LLM est ignoré.")

Support chat: User Jane Doe, marketing manager in Seattle branch of GlobalInc, reported issue with internal tool version 4.2.1 on [DATE_YXC]. Her assistant, based in the same office, added that it's similar to what happened to the finance team in Chicago last quarter. Email: [HOST_CVK]@[HOST_VWT]. Rare combo: marketing role + branch + tool version + cross-team reference. Amount affected: $25000. He noted the MAC address [IP_SZT] and IP [IP_HWS].
detect_failed: JSONParseError: Model did not return JSON: {
  "entities": [
    {
      "id": "e1",
      "type": "PER",
      "surface": "Jane Doe",
      "start": 20,
      "end": 29,
      "cluster_id": "C1",
      "canonical": "PERSON_1",
      "action": "REPLACE",
      "placeholder_type": "P


In [2]:
if os.getenv("OPENROUTER_API_KEY"):
    sample_text_llm = """PROCEDURE\n\nThe case originated in an application (no. 36110/97) against the Republic of Turkey lodged with the European Commission of Human Rights (“the Commission”) under former Article 25 of the Convention for the Protection of Human Rights and Fundamental Freedoms (“the Convention”) by four Turkish nationals, Mr Galip Yalman, Mr Bahattin Sarısoy, Mr Osman Çağlayan and Mr Yusuf Çamca (“the applicants”), on 29 November 1996.\n\nThe applicants were represented by Mr S. Esmer, a lawyer practising in Ankara. The Turkish Government (“the Government”) did not designate an Agent for the purposes of the proceedings before the Convention institutions.\n\nThe applicants alleged that their case, which commenced in 1989 and terminated in 1996, was not heard within a reasonable time as required by the Convention.\n\nThe application was transmitted to the Court on 1 November 1998, when Protocol No. 11 to the Convention came into force (Article 5 § 2 of Protocol No. 11).\n\nThe application was allocated to the Second Section of the Court (Rule 52 § 1 of the Rules of Court). Within that Section, the Chamber that would consider the case (Article 27 § 1 of the Convention) was constituted as provided in Rule 26 § 1.\n\nOn 1 November 2001 the Court changed the composition of its Sections (Rule 25 § 1). This case was assigned to the newly composed First Section (Rule 52 § 1).\n\nBy a decision of 15 May 2003, the Court declared the application admissible.\n\nThe applicants and the Government each filed observations on the merits (Rule 59 § 1).\n\nTHE FACTS\n\nIn September 1980 the applicants were taken into police custody and were subsequently placed in detention on remand on suspicion of membership of an illegal organisation. The Erzincan Military Public Prosecutor filed a bill of indictment with the Erzincan Martial Law Court (sıkıyönetim askeri mahkemesi) charging the applicants under Article 146 of the Criminal Code with membership of an illegal organisation whose object was to undermine the constitutional order.\n\nOn various dates between 1982 and 1984 the applicants were released pending trial.\n\nOn 13 September 1988 the Erzincan Martial Law Court acquitted the applicants of the charges against them.\n\nOn various dates in March, April and June 1989 the applicants brought individual actions before the Sinop Assize Court against the Treasury, pursuant to Law no. 466. They requested compensation for their unjustified detention on remand.\n\nOn 20 October 1989 the court decided to join the applicants’ cases.\n\nOn 15 December 1993 the Sinop Assize Court awarded non‑pecuniary compensation to the applicants. The applicants appealed, arguing that the amount of compensation awarded was not sufficient.\n\nOn 31 January 1995 the Court of Cassation quashed the judgment of the first instance court.\n\nOn 6 June 1995 the Sinop Assize Court increased the amount of non‑pecuniary compensation.\n\nOn 15 June 1995 the applicants again appealed.\n\nOn 30 May 1996 the Court of Cassation dismissed the applicants’ appeal and upheld the judgment of the assize court.", "anonymized_text": "PROCEDURE\n\nThe case originated in an application (no. 36110/97) against the Republic of Turkey lodged with the European Commission of Human Rights (“the Commission”) under former Article 25 of the Convention for the Protection of Human Rights and Fundamental Freedoms (“the Convention”) by four Turkish nationals, Mr Galip Yalman, Mr Bahattin Sarısoy, Mr Osman Çağlayan and Mr Yusuf Çamca (“the applicants”), on 29 November 1996.\n\nThe applicants were represented by Mr S. Esmer, a lawyer practising in Ankara. The Turkish Government (“the Government”) did not designate an Agent for the purposes of the proceedings before the Convention institutions.\n\nThe applicants alleged that their case, which commenced in 1989 and terminated in 1996, was not heard within a reasonable time as required by the Convention.\n\nThe application was transmitted to the Court on 1 November 1998, when Protocol No. 11 to the Convention came into force (Article 5 § 2 of Protocol No. 11).\n\nThe application was allocated to the Second Section of the Court (Rule 52 § 1 of the Rules of Court). Within that Section, the Chamber that would consider the case (Article 27 § 1 of the Convention) was constituted as provided in Rule 26 § 1.\n\nOn 1 November 2001 the Court changed the composition of its Sections (Rule 25 § 1). This case was assigned to the newly composed First Section (Rule 52 § 1).\n\nBy a decision of 15 May 2003, the Court declared the application admissible.\n\nThe applicants and the Government each filed observations on the merits (Rule 59 § 1).\n\nTHE FACTS\n\nIn September 1980 the applicants were taken into police custody and were subsequently placed in detention on remand on suspicion of membership of an illegal organisation. The Erzincan Military Public Prosecutor filed a bill of indictment with the Erzincan Martial Law Court (sıkıyönetim askeri mahkemesi) charging the applicants under Article 146 of the Criminal Code with membership of an illegal organisation whose object was to undermine the constitutional order.\n\nOn various dates between 1982 and 1984 the applicants were released pending trial.\n\nOn 13 September 1988 the Erzincan Martial Law Court acquitted the applicants of the charges against them.\n\nOn various dates in March, April and June 1989 the applicants brought individual actions before the Sinop Assize Court against the Treasury, pursuant to Law no. 466. They requested compensation for their unjustified detention on remand.\n\nOn 20 October 1989 the court decided to join the applicants’ cases.\n\nOn 15 December 1993 the Sinop Assize Court awarded non‑pecuniary compensation to the applicants. The applicants appealed, arguing that the amount of compensation awarded was not sufficient.\n\nOn 31 January 1995 the Court of Cassation quashed the judgment of the first instance court.\n\n On 6 June 1995 the Sinop Assize Court increased the amount of non‑pecuniary compensation.\n\nOn 15 June 1995 the applicants again appealed.\n\nOn 30 May 1996 the Court of Cassation dismissed the applicants’ appeal and upheld the judgment of the assize court. """
    result_L1 = anonymize_text(sample_text_llm, "scope", "secret", level="L0")
    print(result_L1["text"])
    print(result_L1["audit"]["llm_error"])
    [e for e in result_L1["audit"]["entities"] if e.get("source")=="llm-edit"]  # doit être []
else:
    print("Aucune clé OPENROUTER_API_KEY trouvée: le mode LLM est ignoré.")

2025-09-04 18:11:01.46 INFO in 'deeppavlov.download'['download'] at line 138: Skipped http://files.deeppavlov.ai/v1/ner/ner_conll2003_bert_torch_crf.tar.gz download because of matching hashes
2025-09-04 18:11:04.636 INFO in 'deeppavlov.download'['download'] at line 138: Skipped http://files.deeppavlov.ai/v1/ner/ner_ontonotes_bert_torch_crf.tar.gz download because of matching hashes
2025-09-04 18:11:09.628 INFO in 'deeppavlov.download'['download'] at line 138: Skipped http://files.deeppavlov.ai/v1/ner/ner_ontonotes_bert_mult_torch_crf.tar.gz download because of matching hashes
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 17531.05it/s]
/opt/anaconda3/envs/dpv/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens 

PROCEDURE

The case originated in an application ([CODE_LAV]) against the [LOC_TCZ] lodged with the [ORG_YKN] (“[ORG_AHR]”) under former [CODE_XJK] of the [ORG_EBM] (“the Convention”) by four Turkish nationals, [PER_QPD], [PER_OHX], [PER_WAO] and [PER_XCA] (“the applicants”), on [DATE_BGQ].

[ORG_XVJ] were represented by [PER_XRI]. [PER_FBX], a [ORG_XVP] practising in [LOC_MKJ]. The [ORG_MQQ] (“[ORG_FSS]”) did not designate an [PER_ZNH] for the purposes of the proceedings before the [ORG_HZK].

The [ORG_CSN] alleged that their case, which commenced in 1989 and terminated in 1996, was not heard within a reasonable time as required by the Convention.

The application was transmitted to the [ORG_EVP] on [DATE_UFG], when [CODE_RFU] to the Convention came into force ([CODE_GGY] § 2 of [CODE_RFU]).

The [ORG_VPR] was allocated to the [ORG_BLI] ([CODE_VQS] of the Rules of [ORG_EVP]). Within that Section, the [ORG_FMI] that would consider the case ([CODE_CNE] § 1 of the Convention) was constit

In [4]:
result_L1

{'text': 'PROCEDURE\n\nThe case originated in an application ([CODE_LAV]) against the [LOC_TCZ] lodged with the [ORG_YKN] (“[ORG_AHR]”) under former [CODE_XJK] of the [ORG_EBM] (“the Convention”) by four Turkish nationals, [PER_QPD], [PER_OHX], [PER_WAO] and [PER_XCA] (“the applicants”), on [DATE_BGQ].\n\n[ORG_XVJ] were represented by [PER_XRI]. [PER_FBX], a [ORG_XVP] practising in [LOC_MKJ]. The [ORG_MQQ] (“[ORG_FSS]”) did not designate an [PER_ZNH] for the purposes of the proceedings before the [ORG_HZK].\n\nThe [ORG_CSN] alleged that their case, which commenced in 1989 and terminated in 1996, was not heard within a reasonable time as required by the Convention.\n\nThe application was transmitted to the [ORG_EVP] on [DATE_UFG], when [CODE_RFU] to the Convention came into force ([CODE_GGY] § 2 of [CODE_RFU]).\n\nThe [ORG_VPR] was allocated to the [ORG_BLI] ([CODE_VQS] of the Rules of [ORG_EVP]). Within that Section, the [ORG_FMI] that would consider the case ([CODE_CNE] § 1 of the Con

In [7]:
# Appel HTTP vers l'API Flask (assure-toi d'avoir lancé: python main_eval.py)
import json, requests

API_URL = "http://localhost:8000/anonymize"
try:
    resp = requests.post(API_URL, json={
        "text": "Contacte moi sur jean.dupont@example.com concernant le host dev-23.internal.local",
        "level": "L1"  # change en L1 pour activer LLM côté serveur si configuré
    }, timeout=10)
    if resp.status_code == 200:
        data = resp.json()
        print(json.dumps(data, ensure_ascii=False, indent=2))
    else:
        print("Erreur HTTP", resp.status_code, resp.text[:200])
except Exception as e:
    print("Échec appel API:", e)

{
  "audit": {
    "entities": [
      {
        "end": 40,
        "etype": "MAIL",
        "replacement": "[MAIL_NVO]",
        "source": "regex",
        "start": 17,
        "surface": "jean.dupont@example.com"
      }
    ],
    "risk": {
      "findings": [],
      "recommendations": [],
      "risk_score": 0
    }
  },
  "policy": {
    "amount_binning": "none",
    "date_granularity": "none",
    "ip_policy": "public_private",
    "level": "L1",
    "llm_audit": true,
    "llm_detection": true,
    "llm_paraphrase": true,
    "location_granularity": "city",
    "mapping_retention": "session",
    "max_hardening_rounds": 0,
    "org_policy": "categorize",
    "paraphrase_intensity": 1,
    "placeholder_style": "typed",
    "risk_threshold": 45,
    "scope": "ticket",
    "time_granularity": "none"
  },
  "text": "Contacte moi sur [MAIL_NVO] concernant le host dev-23.internal.local",
  "timings_ms": {
    "total": 6
  }
}


## Notes

- `scope_id` contrôle la stabilité des placeholders pour un même groupe de textes.
- `secret_salt` doit rester privé pour éviter l'inversion des pseudonymes.
- Pour ajouter des entités NER pré-calculées, passe une liste de dicts `{start, end, entity_group}` à `ner_results`.
- En mode L1 sans clé valide, le code retombe implicitement sur L0 (raisonneur inactif).

### Pistes d'amélioration
- Ajout d'un flag `return_mapping` pour récupérer les correspondances originales->placeholders.
- Logging structuré (JSON) des décisions d'anonymisation.
- Batch d'anonymisation / endpoint `/anonymize_batch`.


## le Text Anonymization Benchmark (TAB)